# Agentic Workflow to Construct Responses to Google Reviews

In [480]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pptx import Presentation
from typing import TypedDict
import json
import re
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph
from langgraph.graph import END

import warnings
warnings.filterwarnings('ignore')

In [481]:


# ==========================================================
# Configuration
# ==========================================================

from pathlib import Path

# Root directory of synced OneDrive
ONEDRIVE_ROOT = Path(
    r"C:\Users\jsteve\OneDrive - Burlington"
)

# File types to search (update as needed with different extenstions)
SUPPORTED_DOCUMENTS = {

    ".pptx": "powerpoint",
    ".ppt": "powerpoint",

    ".pdf": "pdf",

    ".docx": "word",
    ".doc": "word",

    ".xlsx": "excel",
    ".xls": "excel"
}

# Number of retrieved slides to send to the LLM
TOP_K = 25

print(f"OneDrive Root: {ONEDRIVE_ROOT}")

OneDrive Root: C:\Users\jsteve\OneDrive - Burlington


In [482]:
# ==========================================================
# Initialize LLM for AI Agentic Nodes
# ==========================================================

llm = ChatOllama(
    model="qwen3:8b",
    temperature=0.2,
)

#----------------------
# Top Hits
#----------------------
TOP_K = 20
TOP_K_FINAL = 8

In [483]:
from typing import TypedDict, List, Dict, Any

class SearchState(TypedDict):

    query: str

    document_catalog: list

    document_chunks: list

    document_summaries: list

    retrieved_docs: list

    ranked_docs: list

    ranked_documents: list

    summary: str

    key_points: list

    key_findings: list

    overall_recommendations: list

    sources: list

    memo_path: str

    messages: list

In [484]:
# ==========================================================
# Logging Configuration
# ==========================================================

import logging
import sys

logger = logging.getLogger("OneDriveSearch")

# Prevent duplicate handlers if notebook cell is rerun
if logger.hasHandlers():
    logger.handlers.clear()

logger.setLevel(logging.INFO)

formatter = logging.Formatter(
    "%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%H:%M:%S"
)

# Console output
console_handler = logging.StreamHandler(sys.stdout)
console_handler.setFormatter(formatter)
logger.addHandler(console_handler)

# Optional log file
file_handler = logging.FileHandler("OneDrive_Search.log")
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

logger.info("Logging initialized.")

09:59:55 | INFO | Logging initialized.


In [485]:
# ==========================================================
# Agent 1 - Discover Supported Documents
# ==========================================================

from pathlib import Path
from datetime import datetime
import hashlib


def discover_documents(state: SearchState):

    logger.info("Starting document discovery...")

    catalog = []

    for extension, doc_type in SUPPORTED_DOCUMENTS.items():

        logger.info(f"Searching for *{extension}")

        for file in ONEDRIVE_ROOT.rglob(f"*{extension}"):

            try:

                stat = file.stat()

                catalog.append({
                    "document_id" : hashlib.md5(
                            f"{file.resolve()}_{stat.st_mtime}".encode("utf-8")
                        ).hexdigest(),

                    "file_name": file.name,

                    "path": str(file),

                    "relative_path": str(file.relative_to(ONEDRIVE_ROOT)),

                    "extension": extension,

                    "document_type": doc_type,

                    "size_mb": round(stat.st_size / (1024**2), 2),

                    "modified": datetime.fromtimestamp(
                        stat.st_mtime
                    ).isoformat(),

                    "created": datetime.fromtimestamp(
                        stat.st_ctime
                    ).isoformat(),

                    # Placeholder until we connect Graph API
                    "onedrive_url": None

                })

            except Exception as e:

                logger.warning(f"Unable to access {file}: {e}")

    catalog = sorted(catalog, key=lambda x: x["path"])

    logger.info(
        f"Discovered {len(catalog)} supported documents."
    )

    return {

        "document_catalog": catalog,

        "messages":
            state.get("messages", [])
            + [f"Discovered {len(catalog)} documents."]
    }

In [486]:
#-------------------
# Helper Functions
# ------------------

# ==========================================================
# Helper - Normalize Text
# ==========================================================

import re

def normalize_text(text: str) -> str:
    """
    Cleans extracted text for consistent indexing.
    """

    if not text:
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Collapse multiple spaces
    text = re.sub(r"[ \t]+", " ", text)

    # Collapse excessive blank lines
    text = re.sub(r"\n{3,}", "\n\n", text)

    return text.strip()

# ==========================================================
# Helper - Build Standardized Document Chunk
# ==========================================================

import hashlib

def build_chunk(
    document: dict,
    chunk_number: int,
    text: str,
    section_title: str = "",
    metadata: dict | None = None
):
    """
    Creates a standardized document chunk used throughout the workflow.
    """

    metadata = metadata or {}

    # Clean text
    text = normalize_text(text)
    section_title = normalize_text(section_title)

    # Build searchable text
    combined_text = "\n".join(
        x for x in [section_title, text] if x
    )

    # Stable chunk ID
    chunk_id = hashlib.md5(
        f"{document['document_id']}_{chunk_number}".encode("utf-8")
    ).hexdigest()

    return {

        # ---------- IDs ----------
        "chunk_id": chunk_id,
        "document_id": document["document_id"],

        # ---------- Document Info ----------
        "document_type": document["document_type"],
        "file_name": document["file_name"],
        "path": document["path"],
        "relative_path": document["relative_path"],
        "extension": document["extension"],

        # ---------- Chunk Info ----------
        "chunk_number": chunk_number,
        "section_title": section_title,

        # ---------- Content ----------
        "text": text,
        "combined_text": combined_text,

        # ---------- Parser Metadata ----------
        "metadata": metadata,

        # ---------- File Metadata ----------
        "modified": document["modified"],
        "created": document["created"],
        "size_mb": document["size_mb"],

        # ---------- Source ----------
        "onedrive_url": document["onedrive_url"],

        # ---------- Citation ----------
        "source": {
            "document": document["file_name"],
            "relative_path": document["relative_path"],
            "chunk_number": chunk_number,
            "onedrive_url": document["onedrive_url"]
        },

        # ---------- Future Pipeline ----------
        "embedding": None,
        "retrieval_score": None,
        "rerank_score": None
    }


# ==========================================================
# PowerPoint Helper - Extract Slide Text
# ==========================================================

def extract_ppt_text(slide):
    """
    Extract all searchable text from a PowerPoint slide.
    """

    text = []

    for shape in slide.shapes:

        # Text boxes, placeholders, etc.
        if hasattr(shape, "text"):

            value = normalize_text(shape.text)

            if value:
                text.append(value)

        # Tables
        elif getattr(shape, "has_table", False):

            for row in shape.table.rows:

                cells = [
                    normalize_text(cell.text)
                    for cell in row.cells
                    if normalize_text(cell.text)
                ]

                if cells:
                    text.append(" | ".join(cells))

    return "\n".join(text)


# ==========================================================
# PowerPoint Helper - Extract Speaker Notes
# ==========================================================

def extract_ppt_notes(slide):
    """
    Extract speaker notes if present.
    """

    try:

        notes = []

        for shape in slide.notes_slide.shapes:

            if hasattr(shape, "text"):

                value = normalize_text(shape.text)

                if value:
                    notes.append(value)

        return "\n".join(notes)

    except Exception:

        return ""

In [487]:
# ==========================================================
# Parser - PDF
# ==========================================================

def parse_pdf(document):

    logger.warning(
        f"PDF parser not yet implemented: {document['file_name']}"
    )

    return []

def parse_word(document):

    logger.warning(
        f"Word parser not yet implemented: {document['file_name']}"
    )

    return []

def parse_excel(document):

    logger.warning(
        f"Excel parser not yet implemented: {document['file_name']}"
    )

    return []

In [488]:
# ==========================================================
# Parser - PowerPoint
# ==========================================================

from pptx import Presentation

def parse_powerpoint(document):

    logger.info(
        f"Parsing PowerPoint | "
        f"{document['file_name']} | "
        f"{document['document_id']}"
    )

    chunks = []

    try:

        prs = Presentation(document["path"])

        for slide_num, slide in enumerate(prs.slides, start=1):

            # ----------------------------
            # Extract title
            # ----------------------------
            title = ""

            if slide.shapes.title:
                title = normalize_text(
                    slide.shapes.title.text
                )

            # ----------------------------
            # Extract content
            # ----------------------------
            body = extract_ppt_text(slide)
            notes = extract_ppt_notes(slide)

            body = normalize_text(body)
            notes = normalize_text(notes)

            combined = "\n".join(
                x for x in [body, notes] if x
            )

            # ----------------------------
            # Build metadata
            # ----------------------------
            metadata = {
                "slide_number": slide_num,
                "has_notes": bool(notes.strip())
            }

            chunk = build_chunk(

                document=document,

                chunk_number=slide_num,

                section_title=title,

                text=combined,

                metadata=metadata

            )

            chunks.append(chunk)

        logger.info(
            f"Extracted {len(chunks)} slides | "
            f"{document['file_name']}"
        )

    except Exception as e:

        logger.exception(
            f"Failed parsing PowerPoint | "
            f"{document['file_name']} | "
            f"{document['document_id']}"
        )

    return chunks

In [489]:
# ==========================================================
# Parser Registry
# ==========================================================

PARSERS = {

    "powerpoint": parse_powerpoint,

    "pdf": parse_pdf,

    "word": parse_word,

    "excel": parse_excel

}

In [490]:
# ==========================================================
# Agent 2 - Parse Documents
# ==========================================================

def parse_documents(state: SearchState):

    logger.info("Starting document parsing...")

    document_chunks = []

    for document in state["document_catalog"]:

        parser = PARSERS.get(document["document_type"])

        if parser is None:

            logger.warning(
                f"No parser registered for "
                f"{document['document_type']}"
            )

            continue

        chunks = parser(document)

        document_chunks.extend(chunks)

    logger.info(
        f"Generated {len(document_chunks)} document chunks."
    )

    return {

        "document_chunks": document_chunks,

        "messages": state.get("messages", [])
        + [f"Generated {len(document_chunks)} chunks."]
    }

In [491]:
#-------------------------
# Retrieval Helper Functions
#-------------------------

def normalize_query(query: str) -> str:

    query = query.lower().strip()

    query = re.sub(r"[^\w\s]", " ", query)

    query = re.sub(r"\s+", " ", query)

    return query
def score_chunk(query_tokens, chunk_text: str) -> float:

    text = chunk_text.lower()

    score = 0

    for token in query_tokens:
        if token in text:
            score += 1

    return score


In [492]:
# ==========================================================
# Agent 3 - Retrieve Relevant Chunks (Lightweight Search)
# ==========================================================

def retrieve_documents(state: SearchState):

    logger.info("Retrieving relevant chunks...")

    query = normalize_query(state["query"])
    query_tokens = query.split()

    scored = []

    for chunk in state["document_chunks"]:

        text = chunk.get("combined_text", "")

        score = score_chunk(query_tokens, text)

        # -------------------------
        # TITLE BOOST (FIXED HERE)
        # -------------------------
        title = chunk.get("section_title", "").lower()

        if any(token in title for token in query_tokens):
            score += 3

        if score > 0:
            scored.append((score, chunk))

    scored.sort(key=lambda x: x[0], reverse=True)

    top_k = scored[:TOP_K]

    retrieved = [
        {
            **chunk,
            "retrieval_score": score
        }
        for score, chunk in top_k
    ]

    logger.info(f"Retrieved {len(retrieved)} relevant chunks.")

    return {
        "retrieved_docs": retrieved,
        "messages": state.get("messages", []) + [
            f"Retrieved {len(retrieved)} relevant chunks."
        ]
    }

In [493]:
#-----------------------
# Agent 4 Prompting
#-----------------------

RERANK_PROMPT = """
You are a relevance ranking system working within an off-price retail organization with a focus on marketing.

Task:
Given a user query and an excerpt from a business presentation,
estimate whether this excerpt is likely to contribute useful evidence
toward answering the user's question.

Do NOT evaluate writing quality.

Evaluate only informational usefulness.

Return JSON only.

Fields:
- score: float (0 to 1)
- reason: short explanation (1 sentence max)

User Query:
{query}

Document Chunk:
{chunk}

Rules:

• Focus on informational value.

• Strategy, conclusions, recommendations, executive summaries,
methodology, findings and decision rationale deserve higher scores.

• Ignore formatting, tables of contents and navigation slides.

• Do not reward simple keyword overlap.

• Prefer chunks that contain substantive evidence.

• Return JSON only.
"""


In [494]:
# ==========================================================
# Agent 4 - LLM Reranker
# ==========================================================

import json

def rerank_documents(state: SearchState):

    logger.info("Starting LLM reranking...")

    query = state["query"]
    docs = state["retrieved_docs"]

    ranked = []

    for doc in docs:

        try:

            prompt = RERANK_PROMPT.format(
                query=query,
                chunk=doc["combined_text"][:3000]  # safety cap
            )

            response = llm.invoke(prompt).content.strip()

            # Remove markdown code fences if present
            response = response.replace("```json", "").replace("```", "").strip()

            result = json.loads(response)

            doc_copy = dict(doc)

            doc_copy["rerank_score"] = result.get("score", 0.0)
            doc_copy["rerank_reason"] = result.get("reason", "")

            ranked.append(doc_copy)

        except Exception as e:

            logger.warning(
                f"Rerank failed for chunk "
                f"{doc.get('chunk_id', 'unknown')}: {e}"
            )

            doc["rerank_score"] = 0.0
            doc["rerank_reason"] = "parse_error"

            ranked.append(doc)

# --------------------------------------------------
    # Consolidate chunks into documents
    # --------------------------------------------------

    from collections import defaultdict

    documents = defaultdict(list)

    for chunk in ranked:
        documents[chunk["document_id"]].append(chunk)

    grouped_documents = []

    for document_id, chunks in documents.items():

        # Restore presentation order
        chunks.sort(key=lambda x: x["chunk_number"])

        first = chunks[0]

        grouped_documents.append({

            "document_id": document_id,

            "file_name": first["file_name"],

            "document_type": first["document_type"],

            "path": first["path"],

            "relative_path": first["relative_path"],

            "onedrive_url": first.get("onedrive_url"),

            # Aggregate score
            "document_score": max(
                chunk["rerank_score"]
                for chunk in chunks
            ),

            "combined_text": "\n\n".join(
                chunk["combined_text"]
                for chunk in chunks
            ),

            "num_chunks": len(chunks),

            "chunks": chunks

        })

    # Highest relevance first
    grouped_documents.sort(
        key=lambda x: x["document_score"],
        reverse=True
    )

    logger.info(
        f"Consolidated into "
        f"{len(grouped_documents)} documents."
    )

    return {

        "ranked_docs": grouped_documents,

        "messages": state.get("messages", []) + [

            f"Consolidated {len(ranked)} chunks into "
            f"{len(grouped_documents)} documents."

        ]
    }

In [495]:
DOCUMENT_SUMMARY_PROMPT = """
You are a senior retail strategy analyst at an off-price retailer within the marketing department.

You are reviewing ONE internal business presentation.

All of the excerpts below originate from the SAME document.

Your objective is to reconstruct the narrative of the presentation.

Do NOT summarize each slide.

Instead determine:

• What business problem is being addressed?

• What analysis was performed?

• What evidence was presented?

• What conclusions were reached?

• What recommendations were made?

Write an executive briefing.

Avoid copying sentences from the slides.

Instead synthesize the information into concise business language.

If the retrieved excerpts are incomplete, acknowledge the limitation.

Return ONLY valid JSON.

{{
    "summary":"...",

    "major_findings":[
        "...",
        "...",
        "..."
    ],

    "recommendations":[
        "...",
        "..."
    ]
}}

User Query

{query}

Document Name

{file_name}

Relevant Document Content

{document}
"""

In [496]:
# ==========================================================
# Agent 5 - Document Summarizer (Robust)
# ==========================================================

import json
import re


def extract_json(text: str):
    """
    Attempts to extract the first JSON object from a model response.
    """

    text = text.strip()

    # Remove markdown fences
    text = (
        text.replace("```json", "")
            .replace("```", "")
            .strip()
    )

    # Already valid JSON?
    try:
        return json.loads(text)
    except Exception:
        pass

    # Find first JSON object
    match = re.search(r"\{.*\}", text, re.DOTALL)

    if match:
        return json.loads(match.group())

    raise ValueError("No JSON object found.")


def generate_summary(state: SearchState):

    logger.info("Generating document summaries...")

    query = state["query"]

    documents = state.get("ranked_docs", [])[:TOP_K_FINAL]

    if not documents:

        logger.warning("No documents available.")

        return {
            "document_summaries": [],
            "messages": state.get("messages", []) + [
                "No relevant documents found."
            ]
        }

    summaries = []

    for document in documents:

        logger.info(
            f"Summarizing: {document['file_name']}"
        )

        try:

            # ------------------------------------------
            # Build structured document
            # ------------------------------------------

            slides = []

            for chunk in sorted(
                document["chunks"],
                key=lambda x: x["chunk_number"]
            ):

                slides.append({

                    "slide_number":
                        chunk["chunk_number"],

                    "section_title":
                        chunk.get("section_title", ""),

                    # Keep prompt manageable
                    "content":
                        chunk.get("combined_text", "")[:1200]

                })

            document_payload = {

                "file_name":
                    document["file_name"],

                "document_type":
                    document["document_type"],

                "slides":
                    slides

            }

            logger.info(
                f"Sending {len(slides)} slides to LLM."
            )

            prompt = DOCUMENT_SUMMARY_PROMPT.format(

                query=query,

                file_name=document["file_name"],

                document=json.dumps(
                    document_payload,
                    indent=2
                )

            )

            response = (
                llm.invoke(prompt)
                .content
                .strip()
            )

            logger.info(
                f"LLM response length: {len(response)}"
            )

            result = extract_json(response)

            summaries.append({

                "document_id":
                    document["document_id"],

                "file_name":
                    document["file_name"],

                "relative_path":
                    document["relative_path"],

                "document_type":
                    document["document_type"],

                "document_score":
                    document.get(
                        "document_score",
                        0.0
                    ),

                "onedrive_url":
                    document.get(
                        "onedrive_url"
                    ),

                "summary":
                    result.get(
                        "summary",
                        ""
                    ),

                "major_findings":
                    result.get(
                        "major_findings",
                        []
                    ),

                "recommendations":
                    result.get(
                        "recommendations",
                        []
                    ),

                "sources": [

                    {

                        "chunk_number":
                            chunk["chunk_number"],

                        "section_title":
                            chunk.get(
                                "section_title",
                                ""
                            ),

                        "retrieval_score":
                            chunk.get(
                                "retrieval_score",
                                0.0
                            ),

                        "rerank_score":
                            chunk.get(
                                "rerank_score",
                                0.0
                            ),

                        "relative_path":
                            document["relative_path"],

                        "document_type":
                            document["document_type"],

                        "onedrive_url":
                            document.get(
                                "onedrive_url"
                            ),
                            'path': document["path"]

                    }

                    for chunk in document["chunks"]

                ]

            })

            logger.info(
                f"Completed summary for {document['file_name']}"
            )

        except Exception as e:

            logger.exception(
                f"Failed summarizing {document['file_name']}"
            )

            # Print the raw model output if available
            try:
                logger.error(response)
            except Exception:
                pass

            summaries.append({

                "document_id":
                    document["document_id"],

                "file_name":
                    document["file_name"],

                "relative_path":
                    document.get(
                        "relative_path",
                        ""
                    ),

                "document_type":
                    document.get(
                        "document_type",
                        ""
                    ),

                "document_score":
                    document.get(
                        "document_score",
                        0.0
                    ),

                "onedrive_url":
                    document.get(
                        "onedrive_url"
                    ),

                "summary":
                    "Summary unavailable.",

                "major_findings": [],

                "recommendations": [],

                "sources": []

            })

    logger.info(
        f"Generated summaries for {len(summaries)} documents."
    )

    return {

        "document_summaries":
            summaries,

        "messages":
            state.get("messages", []) + [

                f"Generated {len(summaries)} document summaries."

            ]

    }

In [497]:
# ==========================================================
# Agent 6 Prompt
# ==========================================================

ENTERPRISE_SYNTHESIS_PROMPT = """
You are a senior strategy consultant preparing an executive briefing at a major off-price retailer.

You have already received summaries of several internal business presentations.

Your job is NOT to summarize each presentation again.

Instead, synthesize the information across all documents.

Each document contains:

executive summary
major findings
recommendations

Compare these across documents to identify the following objectives:


• Answer the user's question directly.

• Identify common themes across documents.

• Identify differences or conflicting conclusions.

• Highlight recurring recommendations.

• Point out gaps where information is insufficient.

• Produce insights that would help an executive make decisions.

Do NOT copy wording from the document summaries.

Write concise business prose.

Return ONLY valid JSON.

{{

    "executive_summary":"...",

    "key_findings":[

        {{

            "title":"...",

            "insight":"...",

            "business_impact":"...",

            "supporting_documents":[
                "...",
                "..."
            ]

        }}

    ],

    "overall_recommendations":[

        "...",

        "..."

    ]

}}

User Query

{query}

Document Summaries

{summaries}
"""

In [498]:
# ==========================================================
# Agent 6 - Enterprise Knowledge Synthesizer
# ==========================================================

import json

def synthesize_documents(state: SearchState):

    logger.info("Generating enterprise synthesis...")

    query = state["query"]

    document_summaries = state.get(
        "document_summaries",
        []
    )

    if not document_summaries:

        logger.warning("No document summaries available.")

        return {

            "summary": "No relevant information found.",

            "key_findings": [],

            "overall_recommendations": [],

            "messages": state.get("messages", []) + [
                "No document summaries available."
            ]

        }

    try:

        prompt = ENTERPRISE_SYNTHESIS_PROMPT.format(

            query=query,

            summaries=json.dumps(
                document_summaries,
                indent=2
            )

        )

        response = (
            llm.invoke(prompt)
            .content
            .strip()
        )

        response = (
            response
            .replace("```json", "")
            .replace("```", "")
            .strip()
        )

        logger.info("===== ENTERPRISE RESPONSE =====")
        logger.info(response)
        logger.info("===============================")

        result = extract_json(response)

        logger.info(
            "Enterprise synthesis complete."
        )

        return {

            "summary":
                result.get(
                    "executive_summary",
                    ""
                ),

            "key_findings":
                result.get(
                    "key_findings",
                    []
                ),

            "overall_recommendations":
                result.get(
                    "overall_recommendations",
                    []
                ),

            "document_summaries": [
                {
                    "document_id": d.get("document_id"),
                    "file_name": d.get("file_name"),
                    "relative_path": d.get("relative_path"),
                    "summary": d.get("summary", ""),
                    "key_findings": d.get("major_findings", d.get("key_findings", [])),
                    "recommendations": d.get("recommendations", []),
                    "sources": d.get("sources", [])
                }
                for d in document_summaries
            ],

            "messages":
                state.get("messages", []) + [
                    "Generated enterprise synthesis."
                ]

        }

    except Exception as e:

        logger.exception(e)

        return {

            "summary": f"Enterprise synthesis failed: {e}",

            "key_findings": [],

            "overall_recommendations": [],

            "document_summaries": document_summaries,

            "messages": state.get("messages", []) + [
                f"Enterprise synthesis failed: {e}"
            ]

        }

In [499]:
#------------------
# Agent 7 Helper Functions
#----------------------------

from os import link

from docx import Document
from docx.shared import Pt

# ==========================================================
# Word Hyperlink Helper
# ==========================================================

from pathlib import Path
from docx.oxml.shared import OxmlElement, qn
from docx.opc.constants import RELATIONSHIP_TYPE


def add_hyperlink(paragraph, text, target):
    """
    Adds a clickable hyperlink to a Word paragraph.

    Parameters
    ----------
    paragraph : docx.text.paragraph.Paragraph
    text : str
        Display text
    target : str
        Local path or URL
    """

    # Convert local file paths to file:// URI
    if not str(target).startswith(("http://", "https://", "file://")):
        target = Path(target).resolve().as_uri()

    part = paragraph.part

    r_id = part.relate_to(
        target,
        RELATIONSHIP_TYPE.HYPERLINK,
        is_external=True
    )

    hyperlink = OxmlElement("w:hyperlink")
    hyperlink.set(qn("r:id"), r_id)

    new_run = OxmlElement("w:r")

    rPr = OxmlElement("w:rPr")

    color = OxmlElement("w:color")
    color.set(qn("w:val"), "0563C1")
    rPr.append(color)

    underline = OxmlElement("w:u")
    underline.set(qn("w:val"), "single")
    rPr.append(underline)

    new_run.append(rPr)

    text_element = OxmlElement("w:t")
    text_element.text = text

    new_run.append(text_element)

    hyperlink.append(new_run)

    paragraph._p.append(hyperlink)

def create_memo_doc(
    query: str,
    executive_summary: str,
    key_findings: list,
    recommendations: list,
    document_summaries: list
):

    doc = Document()

    # ==========================================================
    # Title
    # ==========================================================
    doc.add_heading("Query Summary", level=1)
    doc.add_paragraph(f"Query: {query}")
    doc.add_paragraph("")

    # ==========================================================
    # Table of Contents
    # ==========================================================
    doc.add_heading("Table of Contents", level=2)

    toc_items = [
        "Executive Summary",
        "Key Findings",
        "Supporting Insights",
        "Source Appendix"
    ]

    for item in toc_items:
        doc.add_paragraph(item, style="List Number")

    doc.add_paragraph("")

    # ==========================================================
    # Executive Summary
    # ==========================================================
    doc.add_heading("Executive Summary", level=2)

    doc.add_paragraph(executive_summary or "No summary available.")

    doc.add_paragraph("")

    # ==========================================================
    # Key Findings
    # ==========================================================
    doc.add_heading("Key Findings", level=2)

    for i, finding in enumerate(key_findings or [], start=1):

        p = doc.add_paragraph()

        p.add_run(f"{i}. {finding.get('title', 'Untitled')}").bold = True

        p.add_run(f"\n{finding.get('insight', '')}")

        if finding.get("business_impact"):
            p.add_run("\nBusiness Impact: ").bold = True
            p.add_run(finding["business_impact"])

        if finding.get("supporting_documents"):
            p.add_run("\nSupporting Documents: ").bold = True
            p.add_run(", ".join(finding["supporting_documents"]))

        doc.add_paragraph("")

    # ==========================================================
    # Supporting Insights
    # ==========================================================
    doc.add_heading("Supporting Insights", level=2)

    for doc_item in document_summaries or []:

        doc.add_paragraph(
            f"• {doc_item.get('file_name','Unknown')} | "
            f"{doc_item.get('relative_path','')}"
        )

        doc.add_paragraph(doc_item.get("summary", ""))

        doc.add_paragraph("")

    # ==========================================================
    # References
    # ==========================================================
    from collections import defaultdict

    doc.add_heading("References", level=1)

    documents = defaultdict(list)

    for doc_item in document_summaries or []:
        for src in doc_item.get("sources", []):
            documents[doc_item["file_name"]].append(src)

    for file_name, chunks in sorted(documents.items()):

        if not chunks:
            continue

        first = chunks[0]

        p = doc.add_paragraph()

        p.add_run(file_name).bold = True
        p.add_run(f"\nType: {first.get('document_type','')}")
        p.add_run(f"\nFolder: {first.get('relative_path','')}")

        p.add_run("\nReferenced Sections:")

        for chunk in chunks:

            title = chunk.get("section_title", "")

            if title:
                p.add_run(
                    f"\n   • Chunk {chunk.get('chunk_number','')} ({title})"
                )
            else:
                p.add_run(
                    f"\n   • Chunk {chunk.get('chunk_number','')}"
                )

        link = first.get("onedrive_url") or first.get("path")

        logger.info(f"Creating hyperlink for {file_name}")
        logger.info(f"Target: {link}")

        if link:
            add_hyperlink(p, "Open Document", link)

        doc.add_paragraph()

    return doc

In [500]:
# ==========================================================
# Agent 7 - Memo Generator (Enhanced)
# ==========================================================

from datetime import datetime
import os


def generate_memo(state: SearchState):

    logger.info("Generating enhanced Word memo...")

    query = state["query"]

    summary = (
        state.get("summary")
        or state.get("executive_summary")
        or ""
    )

    key_findings = state.get(
        "key_findings",
        []
    )

    overall_recommendations = state.get(
        "overall_recommendations",
        []
    )

    document_summaries = state.get(
        "document_summaries",
        [])

    try:

        doc = create_memo_doc(

            query=query,

            executive_summary=summary,

            key_findings=key_findings,

            recommendations=overall_recommendations,

            document_summaries=document_summaries

        )

        safe_query = (
            query.replace(" ", "_")
                 .replace("/", "_")
        )

        output_dir = "./output"

        os.makedirs(output_dir, exist_ok=True)

        filename = (
            f"memo_{safe_query}_"
            f"{datetime.now():%Y%m%d_%H%M%S}.docx"
        )

        output_path = os.path.join(
            output_dir,
            filename
        )

        doc.save(output_path)

        logger.info(f"""
        Memo Inputs:
        summary length: {len(summary)}
        key_findings: {len(key_findings)}
        doc_summaries: {len(document_summaries)}
        """)

        logger.info(f"Doc summaries preview: {document_summaries[:1]}")
        
        logger.info(f"Memo generated: {output_path}")

        return {
            "memo_path": output_path,
            "summary": summary,
            "key_findings": key_findings,
            "overall_recommendations": overall_recommendations,
            "document_summaries": document_summaries,
            "messages": state.get("messages", []) + [
                f"Memo generated: {output_path}"
            ]
        }

    except Exception:

        logger.exception("Failed to generate memo")

        return {

            "messages": state.get("messages", []) + [
                "Memo generation failed."
            ]
        }

In [501]:
from langgraph.graph import StateGraph, END
from ollama import generate

# Create the graph

workflow = StateGraph(SearchState)

workflow.add_node("discover_documents", discover_documents)
workflow.add_node('parse_documents', parse_documents)
workflow.add_node('retrieve_documents', retrieve_documents)
workflow.add_node('rerank_documents',rerank_documents)
workflow.add_node('generate_summary', generate_summary)
workflow.add_node('synthesize_documents', synthesize_documents)
workflow.add_node('generate_memo', generate_memo)

workflow.set_entry_point('discover_documents')
workflow.add_edge('discover_documents', 'parse_documents')
workflow.add_edge('parse_documents', 'retrieve_documents')
workflow.add_edge('retrieve_documents','rerank_documents')
workflow.add_edge('rerank_documents', 'generate_summary')
workflow.add_edge('generate_summary', 'synthesize_documents')
workflow.add_edge('synthesize_documents', 'generate_memo')
workflow.add_edge('generate_memo', END)

app = workflow.compile()


In [502]:
# Test Run

def init_state(query: str):
    return {
        "query": query,

        "document_catalog": [],
        "document_chunks": [],
        "document_summaries": [],

        "retrieved_docs": [],
        "ranked_docs": [],
        "ranked_documents": [],

        "summary": "",
        "key_points": [],
        "key_findings": [],
        "overall_recommendations": [],
        "sources": [],

        "memo_path": "",
        "messages": []
    }


result = app.invoke(init_state("Traffic"))

for m in result["messages"]:
    logger.info(m)

09:59:56 | INFO | Starting document discovery...
09:59:56 | INFO | Searching for *.pptx
09:59:57 | INFO | Searching for *.ppt
09:59:58 | INFO | Searching for *.pdf
09:59:58 | INFO | Searching for *.docx
09:59:59 | INFO | Searching for *.doc
10:00:00 | INFO | Searching for *.xlsx
10:00:00 | INFO | Searching for *.xls
10:00:01 | INFO | Discovered 98 supported documents.
10:00:01 | INFO | Starting document parsing...
10:00:01 | INFO | Parsing PowerPoint | Big Boulder Young Adult Patroller (YAP) presentation.pptx | a812cd6ac107f8c5e97406f490875018
10:00:01 | INFO | Extracted 9 slides | Big Boulder Young Adult Patroller (YAP) presentation.pptx
10:00:01 | WARNING | Excel parser not yet implemented: 20260225 Assignment 2.xlsx
10:00:01 | WARNING | Word parser not yet implemented: ANA CAMP Certification.docx
10:00:01 | WARNING | PDF parser not yet implemented: SCCM - WHO, WHAT, HOW Framework Tool.pdf
10:00:01 | INFO | Parsing PowerPoint | 042826 Comp and Non Comp Update.pptx | ed3a2f2cc035f4108